In [1]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import gymnasium as gym
from collections import deque

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
class QNetwork(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(QNetwork, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, action_dim)
        )

    def forward(self, x):
        return self.fc(x)

In [3]:
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        state, action, reward, next_state, done = zip(*random.sample(self.buffer, batch_size))
        return (
            torch.tensor(np.array(state), dtype=torch.float32).to(device),
            torch.tensor(action, dtype=torch.int64).to(device),
            torch.tensor(reward, dtype=torch.float32).to(device),
            torch.tensor(np.array(next_state), dtype=torch.float32).to(device),
            torch.tensor(done, dtype=torch.float32).to(device)
        )

    def __len__(self):
        return len(self.buffer)

In [4]:
class DQNAgent:
    def __init__(self, state_dim, action_dim):
        self.state_dim = state_dim
        self.action_dim = action_dim

        self.q_net = QNetwork(state_dim, action_dim).to(device)
        self.target_net = QNetwork(state_dim, action_dim).to(device)
        self.target_net.load_state_dict(self.q_net.state_dict())

        self.optimizer = optim.Adam(self.q_net.parameters(), lr=1e-3)
        self.memory = ReplayBuffer(10000)

        self.batch_size = 64
        self.gamma = 0.99
        self.epsilon = 1.0
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.995

    def select_action(self, state):
        if random.random() < self.epsilon:
            return random.randint(0, self.action_dim - 1)
        state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)
        with torch.no_grad():
            q_values = self.q_net(state_tensor)
        return torch.argmax(q_values).item()

    def train_step(self):
        if len(self.memory) < self.batch_size:
            return

        states, actions, rewards, next_states, dones = self.memory.sample(self.batch_size)

        q_values = self.q_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)
        with torch.no_grad():
            max_next_q_values = self.target_net(next_states).max(1)[0]
            target_q_values = rewards + (1 - dones) * self.gamma * max_next_q_values

        loss = nn.MSELoss()(q_values, target_q_values)

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay

    def update_target_network(self):
        self.target_net.load_state_dict(self.q_net.state_dict())

In [5]:
env = gym.make('CartPole-v1')
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n

agent = DQNAgent(state_dim, action_dim)
episodes = 300
target_update_freq = 10

for episode in range(episodes):
    state, _ = env.reset()
    total_reward = 0
    done = False

    while not done:
        action = agent.select_action(state)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        agent.memory.push(state, action, reward, next_state, done)
        agent.train_step()

        state = next_state
        total_reward += reward

    if (episode + 1) % target_update_freq == 0:
        agent.update_target_network()

    if (episode + 1) % 20 == 0 or episode == 0:
        print(f"Episode {episode + 1}/{episodes}, Total Reward: {total_reward}, Epsilon: {agent.epsilon:.3f}")

env.close()

Episode 1/300, Total Reward: 14.0, Epsilon: 1.000
Episode 20/300, Total Reward: 9.0, Epsilon: 0.248
Episode 40/300, Total Reward: 13.0, Epsilon: 0.056
Episode 60/300, Total Reward: 207.0, Epsilon: 0.010
Episode 80/300, Total Reward: 234.0, Epsilon: 0.010
Episode 100/300, Total Reward: 159.0, Epsilon: 0.010
Episode 120/300, Total Reward: 175.0, Epsilon: 0.010
Episode 140/300, Total Reward: 193.0, Epsilon: 0.010
Episode 160/300, Total Reward: 274.0, Epsilon: 0.010
Episode 180/300, Total Reward: 500.0, Epsilon: 0.010
Episode 200/300, Total Reward: 204.0, Epsilon: 0.010
Episode 220/300, Total Reward: 358.0, Epsilon: 0.010
Episode 240/300, Total Reward: 243.0, Epsilon: 0.010
Episode 260/300, Total Reward: 453.0, Epsilon: 0.010
Episode 280/300, Total Reward: 281.0, Epsilon: 0.010
Episode 300/300, Total Reward: 204.0, Epsilon: 0.010


In [6]:
test_env = gym.make('CartPole-v1')
state, _ = test_env.reset()
total_reward = 0
done = False

while not done:
    state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        action = torch.argmax(agent.q_net(state_tensor)).item()
    next_state, reward, terminated, truncated, _ = test_env.step(action)
    done = terminated or truncated
    state = next_state
    total_reward += reward

print(f"Evaluation Total Reward: {total_reward}")
test_env.close()

Evaluation Total Reward: 111.0
